# Progressive Ratio Experiment Analysis

`ProgressiveRatioExperiment` is a two-well specialisation of `TwoWellExperiment` designed
for experiments that use a progressive-ratio (PR) reinforcement schedule.

The class hierarchy is:

```
Experiment
├── SingleWellExperiment   — single-well (12 independent wells)
└── TwoWellExperiment      — two-well choice chamber
    ├── HedonicFeedingExperiment   — hedonic (reward-based) feeding
    └── ProgressiveRatioExperiment — progressive-ratio schedules
```

### What is a progressive-ratio experiment?

In a PR schedule the number of licks required to activate the reward (lights on) increases
across successive trials.  The fly is first trained to associate a well with a reward signal
during a fixed-ratio **training phase**.  After training, the ratio escalates and the
experiment records how many licks the fly makes during each lit period.  The **breaking
point** is the last trial at which the fly continued to respond — identified here as the
lights-on period at which `DeltaLicks` first drops to (near) zero.

### Features added on top of `TwoWellExperiment`

| Method | Description |
|---|---|
| `ProgressiveRatioExperiment.load()` | Validates `chamber_size=2` and returns the correct subclass |
| `breaking_point_well(dfm, well, configuration)` | Per-well breaking-point DataFrame (Minutes, CumLicks, DeltaLicks …) |
| `breaking_point_dfm(dfm, configuration)` | Runs `breaking_point_well` for all 12 wells of a DFM |
| `breaking_point_summary(configuration)` | Runs `breaking_point_dfm` for every DFM in the experiment |
| `plot_breaking_point_dfm(dfm, configuration)` | Faceted matplotlib line+point plot, one panel per well |
| `plot_breaking_point_dfm_gg(dfm, configuration)` | Same plot using plotnine (ggplot-style) |

All methods inherited from `TwoWellExperiment` and `Experiment` (QC, feeding summary,
PI plots, etc.) remain available unchanged.

## Project Directory

Specify the path to the experiment project directory.  This is the folder that contains
`flic_config.yaml` and (usually) a `data/` sub-folder with the DFM CSV files.

If you place this notebook inside the experiment folder, `"./"` is all you need.

In [ ]:
from pyflic import Experiment
from IPython.display import display

project_directory = "./"

## Setting Up the Configuration

Before loading an experiment you need a `flic_config.yaml` in the project directory.
Use the **config editor** to create or modify it without writing YAML by hand.

**From the terminal** (recommended — run from inside the project directory):

```bash
pyflic-config
```

**From this notebook** (opens a separate window):

In [ ]:
import subprocess
subprocess.Popen(["pyflic-config"], cwd=str(project_directory))

### Key config settings for `ProgressiveRatioExperiment`

| Key | Value | Notes |
|---|---|---|
| `experiment_type` | `progressive_ratio` | Tells `Experiment.load()` to return `ProgressiveRatioExperiment` |
| `chamber_size` | `2` | Required; the loader raises `ValueError` if any DFM uses a different value |
| `well_names` | e.g. `A: Sucrose` / `B: Water` | Optional labels used on plot axes |

#### About V3 DFM files and training detection

PR experiments use **V3 DFM files** (`DFM<id>_*.csv`).  During training the
firmware encodes the in-training flag by adding 65 536 to the raw lick signal for
the active well.  `ProgressiveRatioExperiment` automatically detects and strips
this offset during loading (`_calculate_progressive_ratio_training`) and stores
the end-of-training time for each well in `dfm.in_training_data`.  All
breaking-point methods use this value automatically — you do not need to set it
manually unless you want to override it.

## Loading the Experiment

`Experiment.load()` reads `flic_config.yaml` and returns a `ProgressiveRatioExperiment`
automatically when `experiment_type: progressive_ratio` is set.  No subclass import is
needed.

```python
exp = Experiment.load(project_directory)   # returns ProgressiveRatioExperiment
```

You can also import and call the class directly if you prefer:

```python
from pyflic.base import ProgressiveRatioExperiment
exp = ProgressiveRatioExperiment.load(project_directory)
```

In [ ]:
exp = Experiment.load(project_directory)

### Inspecting training-end times

After loading, `dfm.in_training_data` holds the minute at which the firmware stopped
flagging each well as "in training".  The breaking-point methods use this value to
discard data from the training phase before computing delta licks.

Inspect it for a single DFM to confirm the training was detected correctly:

In [ ]:
# Replace 1 with the ID of any DFM in your experiment.
dfm = exp.get_dfm(1)
display(dfm.in_training_data)

## Running QC and Basic Analysis

`execute_basic_analysis()` runs the standard four-step pipeline inherited from
`Experiment`:

| Step | Output | Location |
|---|---|---|
| 1 | QC reports — integrity check, data-break detection, simultaneous-feeding and bleeding counts, raw-signal plots | `project_dir/qc/` |
| 2 | Experiment summary text | `project_dir/analysis/summary.txt` |
| 3 | Feeding summary CSV — one row per chamber | `project_dir/analysis/feeding_summary.csv` |
| 4 | Feeding summary plot — box + jitter panels for every metric | `project_dir/analysis/feeding_summary.png` |

In [ ]:
exp.execute_basic_analysis()

## Reviewing QC

Launch the interactive QC viewer to inspect raw signal traces, baselined traces,
cumulative lick plots and binned lick histograms before proceeding to breaking-point
analysis.

```bash
pyflic-qc /path/to/project_directory
```

Or from this notebook:

In [ ]:
import subprocess
subprocess.Popen(["pyflic-qc", str(project_directory)])

In a PR experiment pay particular attention to:

- **Training detection** — confirm that `in_training_data` (shown above) captures the
  correct end-of-training minute for each well.  A well that shows no training offset in
  the raw signal will have `NaN` for its training end time; `breaking_point_well` then
  falls back to `end_training=0.0` and includes all post-time-zero data.
- **Lights signal** — check that the `OptoCol1` column is populated and that
  lights-on/off transitions are visible in the baselined trace.
- **Dead or inactive wells** — wells with no response after training are legitimate
  (that is the breaking point), but a well with zero licks throughout training may
  indicate a hardware or placement problem.

## Breaking-Point Analysis — Choosing a Configuration

The `configuration` parameter tells the analysis which well was used as the
**training reference** — the well whose end-of-training time is shared by a group of
adjacent wells.  This reflects the physical wiring of the opto board:

| `configuration` | Training reference wells | Used for wells … |
|---|---|---|
| `1` | W1, W5, W9 | 1–4, 5–8, 9–12 |
| `2` | W3, W7, W11 | 1–4, 5–8, 9–12 |
| `3` | W2, W6, W10 | 1–4, 5–8, 9–12 |
| `4` | W4, W8, W12 | 1–4, 5–8, 9–12 |

In each group of four wells only one well received the PR training signal; the others
share the same training-end minute.  Check your hardware setup or `flic_config.yaml`
notes to confirm which configuration applies to your experiment.

Set the configuration for the cells below:

In [ ]:
# Set to 1, 2, 3, or 4 to match your hardware wiring.
configuration = 1

## Per-Well Breaking-Point Data

`breaking_point_well(dfm, well, configuration)` returns a DataFrame with one row
per **lights-on period** after training:

| Column | Description |
|---|---|
| `Minutes` | Time of the lights-on transition |
| `Lights` | Always `True` (off periods are dropped) |
| `CumLicks` | Cumulative licks from the start of the experiment up to this point |
| `DeltaMinutes` | Duration since the previous lights-on transition |
| `DeltaLicks` | Licks accumulated during this lights-on period |

`DeltaLicks` is the primary measure of interest.  As the ratio escalates, `DeltaLicks`
typically rises then falls; the last trial with a large value is the breaking point.

In [ ]:
# Inspect breaking-point data for a single well.
# Replace dfm_id and well as needed.
dfm = exp.get_dfm(1)
bp_w1 = exp.breaking_point_well(dfm, well=1, end_training=None)
display(bp_w1)

## Per-DFM Breaking-Point Data

`breaking_point_dfm(dfm, configuration)` runs `breaking_point_well` for all 12 wells
at once and returns a dict mapping well name to DataFrame.  This is the most convenient
way to inspect a complete DFM.

In [ ]:
dfm = exp.get_dfm(1)
bp = exp.breaking_point_dfm(dfm, configuration)

# Show the first few rows for each well.
for well_name, df in bp.items():
    print(f"\n--- {well_name} ({len(df)} lights-on periods) ---")
    display(df.head())

## Experiment-Wide Breaking-Point Summary

`breaking_point_summary(configuration)` runs `breaking_point_dfm` for every DFM in the
experiment and returns a nested dict:

```python
{
    dfm_id: {
        "W1": DataFrame,
        "W2": DataFrame,
        ...
    },
    ...
}
```

Use this when you want to aggregate results across all DFMs — for example, to compute
the mean breaking-point trial number per treatment group.

In [ ]:
summary = exp.breaking_point_summary(configuration)

# Example: collect the last DeltaLicks value (a simple proxy for breaking point) per well.
import pandas as pd

rows = []
for dfm_id, wells in summary.items():
    for well_name, df in wells.items():
        if len(df) > 0:
            rows.append({
                "DFM": dfm_id,
                "Well": well_name,
                "Trials": len(df),
                "LastDeltaLicks": float(df["DeltaLicks"].iloc[-1]),
                "MaxDeltaLicks": float(df["DeltaLicks"].max()),
            })

bp_table = pd.DataFrame(rows)
display(bp_table)

## Plotting — Matplotlib Version

`plot_breaking_point_dfm(dfm, configuration)` produces a faceted grid of
`DeltaLicks` vs `Minutes` plots — one panel per well.  This is a direct port of
the plotting loop at the bottom of `breaking_point.R`.

| Parameter | Default | Description |
|---|---|---|
| `ylim` | `(0, 500)` | Y-axis limits; pass `None` for auto-scaling |
| `ncols` | `4` | Number of columns in the facet grid |

In [ ]:
import matplotlib.pyplot as plt

dfm = exp.get_dfm(1)
fig = exp.plot_breaking_point_dfm(dfm, configuration, ylim=(0, 500), ncols=4)
plt.show()

## Plotting — Plotnine (ggplot) Version

`plot_breaking_point_dfm_gg(dfm, configuration)` produces the same faceted plot using
[plotnine](https://plotnine.readthedocs.io/), the Python ggplot port.  It returns a
plotnine `ggplot` object so you can further customise it with `+` expressions or save
it with `.save()`.

| Parameter | Default | Description |
|---|---|---|
| `ylim` | `(0, 500)` | Y-axis limits via `coord_cartesian`; pass `None` for auto |
| `ncols` | `4` | Columns in the facet grid |
| `point_size` | `2.0` | Size of the points |
| `line_size` | `0.6` | Width of the connecting lines |
| `base_font_size` | `10.0` | Base font size for `theme_bw` |
| `figsize` | auto | `(width, height)` in inches |

In [ ]:
dfm = exp.get_dfm(1)
p = exp.plot_breaking_point_dfm_gg(dfm, configuration, ylim=(0, 500))
p

### Saving the plotnine figure

```python
p.save("breaking_point_dfm1.png", dpi=150)
```

Or loop over all DFMs and save one file per DFM:

In [ ]:
from pathlib import Path

out_dir = Path(project_directory) / "analysis" / "breaking_point"
out_dir.mkdir(parents=True, exist_ok=True)

for dfm_id, dfm in exp.dfms.items():
    p = exp.plot_breaking_point_dfm_gg(dfm, configuration)
    p.save(str(out_dir / f"breaking_point_dfm{dfm_id}.png"), dpi=150, verbose=False)
    print(f"  Saved DFM {dfm_id}")

print(f"Done — plots written to {out_dir}")

## Overriding the Training End Time

If the automatic training-end detection produces an incorrect value for a specific well
(e.g. because the training flag was not set in that file), you can pass `end_training`
manually to `breaking_point_well`:

```python
# Manually set training end to 45 minutes for well 3.
df = exp.breaking_point_well(dfm, well=3, end_training=45.0)
```

Pass `end_training=0.0` to include all data (skip the training filter entirely).

`breaking_point_dfm` and `breaking_point_summary` always derive training-end times
automatically from `dfm.in_training_data`; use `breaking_point_well` directly when
you need per-well overrides.

## Standard TwoWellExperiment Methods

Because `ProgressiveRatioExperiment` inherits from `TwoWellExperiment`, all standard
two-well analysis methods are available alongside the breaking-point methods:

```python
# Feeding summary table
df = exp.feeding_summary()

# Well-duration jitter plot (WellA vs WellB per treatment)
p = exp.facet_plot_well_durations(metric="MedDuration")

# Cumulative PI plot for a single DFM / chamber
exp.plot_cumulative_licks_chamber(dfm_id=1, chamber=1)

# Binned licks over time by treatment group
exp.plot_binned_licks_by_treatment()
```